# 레슨 3-2. 규제형 로지스틱 CSS와 스코어카드

**예상 시간** 90분 · **선행** 레슨 2-2, 3-1 · **이론 페이지** `docs/theory/stage_07.html`

## 학습 목표
1. **점수 = Offset + Factor × ln(odds_good)** 의 의미와 PDO(오즈가 2배가 되는 점수)를 계산한다.
2. WOE 변수로 로지스틱 모형을 만들고 **구간별 점수표(스코어카드)** 로 바꾼다.
3. 점수표의 합계가 모형과 **정확히 일치하는지 검산**하고, 반올림 오차의 한계를 안다.
4. 점수 구간을 **등급**으로 나누고 등급별 건수·부도율·신뢰구간·단조성을 확인한다.
5. **컷오프는 정책**이며 예측모형과 분리해서 다룬다는 것을 이해한다.

> **약속**: 점수가 높을수록 안전합니다. 모든 경계·WOE·계수는 **개발 구간**에서 정하고 테스트에는 적용만 합니다.
> 이 레슨은 교육용입니다. 실제 은행의 내부등급법(IRB)·감독규정 적합성을 인증하지 않습니다.

In [ ]:
# [환경 준비] 맨 위 셀부터 순서대로 실행하세요. (Colab/로컬 모두 동작)
import os
import subprocess
import sys
import warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/nh15312323-art/CreditModel.git"


def find_or_fetch_project() -> Path:
    for base in (Path.cwd(), Path.cwd().parent, Path("/content/credit-model-mastery-lab")):
        if (base / "src" / "creditlab").exists():
            return base
    if not IN_COLAB:
        raise RuntimeError("프로젝트 폴더를 찾지 못했습니다. 저장소 루트(또는 notebooks/)에서 노트북을 여세요.")
    if REPO_URL:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/content/credit-model-mastery-lab"], check=True)
    else:
        from google.colab import files  # Colab 전용

        print("credit-model-mastery-lab.zip 파일을 선택해 업로드하세요.")
        files.upload()
        subprocess.run(["unzip", "-q", "-o", "credit-model-mastery-lab.zip", "-d", "/content"], check=True)
    return Path("/content/credit-model-mastery-lab")


PROJECT_ROOT = find_or_fetch_project()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))
if IN_COLAB:  # UCI 원본이 엑셀(.xls)이라 읽기 엔진이 필요합니다
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "xlrd", "openpyxl"], check=False)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import creditlab
from creditlab import environment_info, set_seed

SEED = set_seed(42)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
print("프로젝트 위치:", PROJECT_ROOT)
print("creditlab", creditlab.__version__, "| 시드", SEED)
print({k: v for k, v in environment_info(SEED)["packages"].items()})

## 1. 개념 — 오즈를 점수로

`odds_good = (1 − PD) / PD` (정상:부도). 점수는 이 로그의 **선형 변환**입니다.

`Score = Offset + Factor × ln(odds_good)`, `Factor = PDO / ln 2`, `Offset = 기준점수 − Factor × ln(기준 오즈)`

예: **600점일 때 50:1, PDO 20** 이면 점수가 20점 오를 때마다 오즈가 2배가 됩니다.

In [ ]:
pred_score_100 = None  # ← 오즈가 100:1 이면 점수는? (힌트: PDO 의 뜻)  25:1 이면?

In [ ]:
from creditlab.scorecard import ScoreScaling

sc = ScoreScaling(base_score=600, base_odds=50, pdo=20)
print(f"Factor = 20/ln2 = {sc.factor:.4f}, Offset = 600 − Factor×ln50 = {sc.offset:.4f}")

pd_grid = np.array([0.005, 0.01, 0.02, 0.05, 0.10, 0.20])
tbl = pd.DataFrame({"PD": pd_grid, "odds(good:bad)": (1 - pd_grid) / pd_grid, "점수": sc.score_from_pd(pd_grid)})
print(tbl.round(3).to_string(index=False))
print(f"\n검산: 50:1 → {float(sc.score_from_odds_good(50)):.1f}점, 100:1 → {float(sc.score_from_odds_good(100)):.1f}점, 25:1 → {float(sc.score_from_odds_good(25)):.1f}점")
print(f"600점의 PD = {float(sc.pd_from_score(600)):.4f} (= 1/51 = {1/51:.4f})")
if pred_score_100 is not None:
    print("내 예측:", pred_score_100)

**주의**: PD 가 정확히 1/51 이어도 점수 600 은 *"기준 오즈를 50:1 로 정했기 때문"* 입니다. 기준점·기준 오즈·PDO 는 **설계 선택**이므로 문서에 적습니다.

## 2. 데이터, 비닝, 변수 선별

In [ ]:
from creditlab.data import load_learning_data
from creditlab.features import fit_binners, iv_summary, transform_woe_frame

df_all, meta = load_learning_data("synthetic", seed=SEED)
TARGET = meta["target"]
df_all = df_all.sort_values("application_date").reset_index(drop=True)
cut = int(len(df_all) * 0.7)
dev, test = df_all.iloc[:cut].copy(), df_all.iloc[cut:].copy()

CANDS = ["age", "income_10k", "loan_amount_10k", "employment_years", "card_utilization", "num_delinq_12m", "inquiries_6m", "housing_type", "region"]
rank = iv_summary(dev, CANDS, TARGET, method="quantile", max_bins=5, min_events=10)
print(rank[["variable", "n_bins", "iv", "iv_label", "bad_rate_trend"]].round(4).to_string(index=False))

IV 는 **선별 참고용**입니다. 여기서는 *IV 0.02 이상 + 업무적으로 설명 가능*한 변수만 남깁니다. 지역(`region`)처럼 보호 속성의 대리 변수가 될 수 있는 변수는 IV 와 별개로 법적·공정성 검토가 필요합니다(레슨 2-1). 이렇게 정한 선별 규칙은 **개발 구간의 숫자로만** 만들고 문서화합니다.

In [ ]:
SELECTED = rank.loc[rank["iv"] >= 0.02, "variable"].tolist()
print("선택된 변수:", SELECTED)
print("제외된 변수:", [c for c in CANDS if c not in SELECTED])

binners = fit_binners(dev, SELECTED, TARGET, method="quantile", max_bins=5, min_events=10)
Wd = transform_woe_frame(dev, binners)
Wt = transform_woe_frame(test, binners)           # 테스트에는 transform 만
print(Wd.shape, Wt.shape)

## 3. WOE 로지스틱 모형 — 계수 부호와 크기 점검

In [ ]:
from creditlab.models import fit_logistic

fit = fit_logistic(Wd, dev[TARGET])
print(fit.summary()[["variable", "coef", "std_err", "p_value", "odds_ratio"]].round(4).to_string(index=False))
print("\n경고:", fit.warnings or "없음", "| 수렴:", fit.converged)
neg = [n for n, c in fit.slopes.items() if c < 0]
print("음수 계수 변수:", neg or "없음", "← WOE 부호 약속(높을수록 위험) 때문에 대체로 모두 양(+)이어야 정상")

- WOE 입력의 계수는 **1 근처**가 자연스럽습니다(WOE 가 이미 로그오즈 단위). 크게 벗어나면 그 변수의 비닝이 불안정하거나 다른 변수와 겹치는 신호입니다.
- 음수 계수가 나오면 *점수 방향이 업무 논리와 반대*가 됩니다. 원인을 확인하기 전에는 모형에 쓰지 않습니다(공선성·비닝 불안정).

## 4. 스코어카드 만들기와 검산

In [ ]:
from creditlab.scorecard import Scorecard

card = Scorecard.from_fit(binners, fit, scaling=sc)
t = card.table_.copy()
print(f"사용 변수 {len(card.variables)}개, 기준점수 몫(변수당): {card._base_share:.2f}")
print(t[t["variable"] == "card_utilization"][["variable", "bin", "n", "bad_rate", "woe", "points"]].round(3).to_string(index=False))

**점수표 읽는 법**: 구간의 WOE 가 클수록(위험) 점수는 낮습니다. 신청자의 점수는 **변수별 구간 점수의 합**입니다. 변수당 기준점수 몫 `(Offset − Factor·β0)/m` 이 모든 구간 점수에 더해져 있습니다.

In [ ]:
for name, d in [("개발", dev), ("테스트", test)]:
    v = card.verify(d)
    print(f"[{name}] 점수표 합계 vs 모형(Offset − Factor·logit) 최대 차이 = {v['max_abs_diff']:.2e}  (허용 {v['allowed']:.0e}) → {'일치' if v['ok'] else '불일치!'}")

card_int = Scorecard.from_fit(binners, fit, scaling=sc, round_points=True)
v2 = card_int.verify(test)
print(f"\n[정수 점수로 반올림] 최대 차이 = {v2['max_abs_diff']:.2f}점 (이론 한계 0.5×변수 수 = {v2['allowed']:.1f}점)")

s_test = card.score(test)
print(f"\n테스트 점수 요약: 최소 {s_test.min():.0f}, 중앙 {s_test.median():.0f}, 최대 {s_test.max():.0f}")
print("점수에서 되돌린 PD 와 모형 PD 의 최대 차이:", float(np.abs(sc.pd_from_score(s_test) - card.predict_pd(test)).max()))

점수표는 모형을 **구간 점수로 옮긴 것**이라 합계가 모형과 정확히 같아야 합니다. 정수로 반올림하면 신청자마다 최대 `0.5×변수 수` 점까지 어긋날 수 있으니, 그 오차를 문서에 남기고 PD 환산은 반올림 전 값 또는 점수표 기준 중 하나로 **일관되게** 합니다.

## 5. 점수 → PD 관계 확인

In [ ]:
bands = pd.cut(card.score(dev), bins=[-np.inf, 520, 540, 560, 580, 600, np.inf])
chk = pd.DataFrame({"band": bands, "default": dev[TARGET], "pd_model": card.predict_pd(dev)}).groupby("band", observed=True).agg(
    n=("default", "size"), 실제부도율=("default", "mean"), 모형PD평균=("pd_model", "mean"))
print(chk.round(4).to_string())
fig, ax = plt.subplots(figsize=(5, 3.5))
ax.plot(sorted(card.score(dev)), sorted(card.predict_pd(dev), reverse=True), color="#c8553d")
ax.set_xlabel("score"); ax.set_ylabel("model PD"); ax.set_title("score vs PD (monotone decreasing)")
plt.show()

점수가 높을수록 PD 는 **단조 감소**합니다. 개발 구간에서는 구간별 실제 부도율과 모형 PD 평균이 가깝습니다(개발 데이터에 맞췄으므로 당연). **테스트에서도 그런지**는 Phase 4(보정·PSI)에서 정식으로 검증합니다.

## 6. 등급화 — 동일 건수 · 동일 폭 · 업무 기준

In [ ]:
pred_width = None  # ← 점수 폭을 똑같이 나눈 5등급(width)에서는 건수가 고르게 나뉠까요? 어느 등급이 가장 적을까요?

In [ ]:
from creditlab.scorecard import grade_monotonic, grade_table, make_grade_edges

s_dev, s_te = card.score(dev), card.score(test)
edges_q = make_grade_edges(s_dev, 5, "quantile")            # 개발 구간 점수로만 경계를 만든다
edges_w = make_grade_edges(s_dev, 5, "width")
edges_m = [520, 550, 570, 590]                                # 업무 기준(사람이 정한 경계)

summary = []
for name, e in [("동일 건수", edges_q), ("동일 폭", edges_w), ("업무 기준", edges_m)]:
    tb = grade_table(s_dev, dev[TARGET], e)
    summary.append({"방식": name, "경계": [round(x) for x in e], "등급별 건수": tb["n"].tolist(), "등급별 부도건수": tb["bad"].tolist(),
                    "개발 부도율 추세": grade_monotonic(tb)})
print(pd.DataFrame(summary).to_string(index=False))
if pred_width:
    print("\n내 예측:", pred_width)

동일 폭 등급은 점수 분포의 꼬리에 **건수·부도가 거의 없는 등급**이 생겨 부도율이 불안정합니다. 등급 설계는 ① 등급별 표본·부도 건수가 충분한가 ② 부도율이 **단조 증가**하는가 ③ 업무적으로 설명 가능한가를 함께 봅니다.

In [ ]:
t_dev = grade_table(s_dev, dev[TARGET], edges_q)
t_te = grade_table(s_te, test[TARGET], edges_q)               # 테스트에는 '개발에서 만든 경계'를 그대로
out = t_dev[["grade", "n", "bad", "bad_rate"]].rename(columns={"n": "개발 n", "bad": "개발 부도", "bad_rate": "개발 부도율"})
out["테스트 n"], out["테스트 부도"], out["테스트 부도율"] = t_te["n"], t_te["bad"], t_te["bad_rate"]
out["테스트 95% 구간"] = [f"{lo:.1%}~{hi:.1%}" for lo, hi in zip(t_te["bad_rate_lo"], t_te["bad_rate_hi"])]
print(out.round(3).to_string(index=False))
print("\n단조성 — 개발:", grade_monotonic(t_dev), "| 테스트:", grade_monotonic(t_te))

**읽는 법**: 테스트 구간의 부도 건수는 등급당 수십 건뿐이라 신뢰구간이 넓습니다. 이웃 등급의 구간이 겹치면 *순서가 뒤집혀 보이는 것*도 우연일 수 있습니다. 한두 등급이 약간 어긋났다고 곧바로 모형을 버리거나, 반대로 경계를 테스트에 맞게 고치면 안 됩니다.

## 7. 컷오프는 정책이다

In [ ]:
pred_cut = None  # ← 컷오프를 580 으로 올리면 승인율은 내려갈까요, 올라갈까요? 승인 건의 부도율은?

In [ ]:
from creditlab.scorecard import cutoff_analysis

ct = cutoff_analysis(s_te, test[TARGET], [500, 530, 550, 570, 580, 590])
print(ct.round(3).to_string(index=False))
if pred_cut:
    print("\n내 예측:", pred_cut)

컷오프를 올리면 승인율이 내려가고 승인 건의 부도율도 내려갑니다(거절되는 부도 비중은 올라감). **어느 지점을 고를지는 모형이 아니라 정책 문제**입니다 — 수익·손실·규제·고객 영향·한도 전략을 함께 봅니다. 그래서 *모형 성능 평가*와 *컷오프 성과 분석*을 섞지 않습니다.

실무에는 이 밖에도 **override**(심사역 수기 조정), **reject**(정책 거절), 수동심사 구간이 있어 모형 점수와 최종 결정이 다를 수 있습니다. 사후 모니터링(Phase 4)에서는 이를 구분해 봅니다.

## 8. ✏️ 실습 — 신청자 한 명의 점수 검산

In [ ]:
# [exercise]
applicant = test.iloc[[0]]
pts = card.points_frame(applicant).iloc[0]
print("신청자 구간별 점수:")
print(pts.round(2).to_string())

my_total = None     # ← 구간별 점수의 합계 (소수 둘째 자리까지)
my_pd = None        # ← 그 점수에 해당하는 부도확률 (점수 → PD 변환식을 써서)
# 힌트1: 합계는 pts.sum().   힌트2: PD = 1 / (1 + odds_good),  odds_good = exp((점수 − Offset)/Factor)  → sc.pd_from_score(점수)

if my_total is not None and my_pd is not None:
    true_total = float(pts.sum())
    true_pd = float(card.predict_pd(applicant)[0])
    print(f"\n합계 {my_total:.2f} (정답 {true_total:.2f}) :", "맞음" if abs(my_total - true_total) < 0.01 else "다시 계산")
    print(f"PD {my_pd:.4f} (모형 PD {true_pd:.4f}) :", "맞음" if abs(my_pd - true_pd) < 1e-4 else "다시 계산")
else:
    print("\nmy_total, my_pd 를 채우고 다시 실행하세요.")

<details>
<summary>정답 예시 (먼저 직접 해 본 뒤 펼치세요)</summary>

```python
my_total = round(float(pts.sum()), 2)
my_pd = float(sc.pd_from_score(my_total))
```
</details>

## 9. 퀴즈

In [ ]:
from creditlab.learning import Question, check, grade, show

QUIZ = [
    Question("Q1", "PDO", "600점이 50:1(good:bad), PDO 가 20 입니다. 640점의 오즈는?",
             {"A": "100:1", "B": "200:1", "C": "70:1", "D": "52:1"},
             "B", "20점마다 오즈가 2배입니다. 600→620 은 100:1, 620→640 은 200:1.", ("20점 올라갈 때마다 오즈가 몇 배가 되나요?",)),
    Question("Q2", "점수 방향", "이 프로그램에서 점수가 높아지면?",
             {"A": "부도확률이 높아진다", "B": "odds_good 이 커지고 부도확률이 낮아진다", "C": "WOE 가 커진다", "D": "등급이 E 쪽으로 간다"},
             "B", "Score = Offset + Factor × ln(odds_good). 점수가 높을수록 안전합니다.", ("odds_good 은 정상:부도 입니다.",)),
    Question("Q3", "점수표 검산", "정수로 반올림한 점수표의 합계가 모형 점수와 최대 2점 어긋났습니다. 변수가 6개입니다. 정상인가요?",
             {"A": "비정상이다. 무조건 오류", "B": "정상이다. 반올림 오차는 최대 0.5×변수 수(=3점) 이내이다",
              "C": "점수표는 원래 모형과 상관없다", "D": "변수를 삭제해야 한다"},
             "B", "구간 점수마다 최대 0.5점씩 반올림되므로 합계 오차의 한계는 0.5×변수 수입니다.", ("변수마다 몇 번 반올림되나요?",)),
    Question("Q4", "등급 설계", "동일 폭 등급에서 A 등급의 부도가 3건뿐입니다. 올바른 대응은?",
             {"A": "A 등급 부도율 0.3% 를 그대로 믿는다", "B": "표본·부도 건수가 부족하니 신뢰구간을 확인하고 등급 경계를 개발 구간에서 다시 설계(건수 기반 등)한다",
              "C": "테스트에서 부도율이 좋아 보이는 경계로 고친다", "D": "A 등급을 삭제한다"},
             "B", "경계는 개발 구간에서 설계합니다. 테스트를 보고 경계를 맞추면 누수입니다.", ("신뢰구간 폭을 보세요.",)),
    Question("Q5", "모형 vs 정책", "승인 컷오프를 580점으로 정했습니다. 이것은?",
             {"A": "예측모형의 성능 지표다", "B": "모형이 아니라 정책 결정이다. 승인율·손실·규제·고객 영향을 함께 보고 따로 승인한다",
              "C": "모형 학습에 쓰이는 하이퍼파라미터다", "D": "AUROC 로 정해진다"},
             "B", "점수는 위험의 순서와 크기를 주는 모형의 출력이고, 어디서 자를지는 정책입니다.", ("같은 모형이어도 회사마다 컷오프는 다를 수 있습니다.",)),
]

answers = {"Q1": None, "Q2": None, "Q3": None, "Q4": None, "Q5": None}  # ← "A"~"D"

for q in QUIZ:
    show(q)
    print()

In [ ]:
for q in QUIZ:
    check(q, answers[q.qid])
if all(v is not None for v in answers.values()):
    r = grade(QUIZ, answers)
    print("\n점수:", r["score"], "/", r["total"], "| 통과(75% 기준):", r["passed"], "| 다시 볼 개념:", r["concepts_to_review"])

## 10. 과제와 정리
**과제**
1. 기준점수 650, 기준 오즈 100:1, PDO 30 으로 `ScoreScaling` 을 바꿔 같은 모형의 점수 분포를 다시 만들고, **등급별 부도율이 변하지 않는지** 확인하세요. (점수 척도는 선형 변환이므로 순서는 같아야 합니다.) 왜 그런지 한 문장으로 쓰세요.
2. L2 규제 모형(`fit_regularized`)으로 스코어카드를 만들어(`Scorecard.from_sklearn`) 규제 없는 모형과 점수가 얼마나 다른지 비교하세요.
3. UCI 데이터로 같은 절차(비닝 → 로지스틱 → 점수표 → 5등급)를 수행하고 **개발 대비 테스트** 등급별 부도율을 표로 정리하세요. 단조성이 깨진 등급이 있다면 신뢰구간으로 우연 여부를 논하세요.

**체크포인트**: 퀴즈 4/5 이상 · 점수 검산 실습 통과 · 점수표 합계와 모형 일치 확인

**다음 Phase**: 검증과 모니터링 — AUROC·Gini·KS·보정·PSI 로 이 모형을 정식으로 평가합니다.

**참고자료**
- Siddiqi, N., *Credit Risk Scorecards* (Wiley, 2006) — 점수 변환(PDO)과 스코어카드.
- Thomas, L., *Consumer Credit Models* (Oxford, 2009).
- 이론: `docs/theory/stage_07.html`